## Camada Prata: Limpeza e Padronização

Recebe o dado bruto da Bronze e aplica:
- Decodificação de hashes (cidades, empresas, clientes)
- Renomeação de colunas para português
- Remoção de registros inconsistentes
- Classificação do período COVID
- Divisão temporal sem data leakage

Entrada : clickbus_bronze.parquet (1.741.344 linhas)

Saída   : clickbus_prata.parquet  (dado limpo, pronto para Ouro)

In [1]:
import pandas as pd
import numpy as np
import sys
from google.colab import drive

drive.mount('/content/drive')

!git clone https://github.com/vsmacedo-datafinance/Challenge_ClickBus_FIAP_2025.git
sys.path.append('/content/Challenge_ClickBus_FIAP_2025')

from src.utils import tratar_dados_clickbus, salvar_mapeamentos

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path 'Challenge_ClickBus_FIAP_2025' already exists and is not an empty directory.


In [2]:
df = pd.read_parquet("/content/drive/MyDrive/Portifólio DS Vini/Challenge_ClickBus_2025/data/bronze/clickbus_bronze.parquet")

print(f"Bronze carregado: {df.shape}")
print(f"Período: {df['date_purchase'].min()} à {df['date_purchase'].max()}")

Bronze carregado: (1741344, 12)
Período: 2013-09-12 à 2024-04-01


## Decodificação de Hashes e Variável COVID

In [3]:
df['date_purchase'] = pd.to_datetime(df['date_purchase'])

df_limpo, dicionarios = tratar_dados_clickbus(df)

A pandemia foi um **"choque exógeno"** que alterou drasticamente os padrões de viagem.

In [4]:
df_limpo.info()
print("-----")
print(f"Cidades mapeadas : {len(dicionarios['cidades'])}")
print(f"Empresas mapeadas: {len(dicionarios['empresas'])}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1741344 entries, 0 to 1741343
Data columns (total 20 columns):
 #   Column                               Dtype         
---  ------                               -----         
 0   nk_ota_localizer_id                  object        
 1   fk_contact                           object        
 2   date_purchase                        datetime64[ns]
 3   time_purchase                        object        
 4   place_origin_departure               object        
 5   place_destination_departure          object        
 6   place_origin_return                  object        
 7   place_destination_return             object        
 8   fk_departure_ota_bus_company         object        
 9   fk_return_ota_bus_company            object        
 10  gmv_success                          float64       
 11  total_tickets_quantity_success       int64         
 12  place_origin_departure_legend        object        
 13  place_destination_departure

In [5]:
df_limpo.head()

,nk_ota_localizer_id,fk_contact,date_purchase,time_purchase,place_origin_departure,place_destination_departure,place_origin_return,place_destination_return,fk_departure_ota_bus_company,fk_return_ota_bus_company,gmv_success,total_tickets_quantity_success,place_origin_departure_legend,place_destination_departure_legend,place_origin_return_legend,place_destination_return_legend,fk_departure_ota_bus_company_legend,fk_return_ota_bus_company_legend,fk_contact_legend,periodo_covid
0,bc02d5245bec63b30ff1102fa273fc03f58bc9cc3f674e...,a7218ff4ee7d37d48d2b4391b955627cb089870b934912...,2018-12-26,15:33:35,6b86b273ff34fce19d6b804eff5a3f5747ada4eaa22f1d...,50e9a8665b62c8d68bccc77c7c92431a1aa26ccbd38ed4...,0,0,8527a891e224136950ff32ca212b45bc93f69fbb801c3b...,1,89.09,1,Cidade_0,Cidade_76,0,0,Empresa_0,Empresa_299,Cliente_380081,pre-covid
1,5432f12612dd5d749b3be880e779989cf63b5efa4bcc4e...,37228485e0dc83d84d1bcd1bef3dc632301bf6cb22c8b5...,2018-12-05,15:07:57,10e4e7caf8b078429bb1c80b1a10118ac6f963eff098fd...,e6d41d208672a4e50b86d959f4a6254975e6fb9b088116...,0,0,36ebe205bcdfc499a25e6923f4450fa8d48196ceb4fa0c...,1,155.97,1,Cidade_1,Cidade_900,0,0,Empresa_1,Empresa_299,Cliente_125030,pre-covid
2,fb3caed9b2f1b6016d45ccddb19095476e61a2c85faa8e...,3467ec081e2421e72c96e7203b929d21927fd00b6b5f28...,2018-12-21,18:41:54,7688b6ef52555962d008fff894223582c484517cea7da4...,8c1f1046219ddd216a023f792356ddf127fce372a72ec9...,0,0,ec2e990b934dde55cb87300629cedfc21b15cd28bbcf77...,1,121.99,1,Cidade_2,Cidade_367,0,0,Empresa_2,Empresa_299,Cliente_118673,pre-covid
3,4dc44a6dd592b702feccb493d192210c86965aee684529...,ab3251a2be0f69713b8f97b0e9d1579e31551f4fd4facf...,2018-12-06,14:01:38,4e07408562bedb8b60ce05c1decfe3ad16b72230967de0...,d6acb3c1a79e57bcc03d976cb4d98f56edccd4cf426392...,0,0,5f9c4ab08cac7457e9111a30e4664920607ea2c115a143...,1,55.22,1,Cidade_3,Cidade_1284,0,0,Empresa_3,Empresa_299,Cliente_389356,pre-covid
4,aa34ed7fd0a6b405df2df1bf9f8d68e6df9b9a868a6181...,ceea0de820a6379f2c4215bddaec66c33994b304607e56...,2021-02-23,20:08:25,7688b6ef52555962d008fff894223582c484517cea7da4...,23765fc69c4e3c0b10f5d15471dc2245e2a19af16b513f...,0,0,48449a14a4ff7d79bb7a1b6f3d488eba397c36ef25634c...,1,45.31,1,Cidade_2,Cidade_65,0,0,Empresa_4,Empresa_299,Cliente_470537,durante-covid


In [6]:
colunas_hash = [
    'fk_contact',
    'place_origin_departure', 'place_destination_departure',
    'place_origin_return',    'place_destination_return',
    'fk_departure_ota_bus_company', 'fk_return_ota_bus_company'
]
df_limpo = df_limpo.drop(columns=colunas_hash)

df_limpo = df_limpo.rename(columns={
    'date_purchase'                       : 'data_compra',
    'fk_contact_legend'                   : 'id_cliente',
    'place_origin_departure_legend'       : 'origem_ida',
    'place_destination_departure_legend'  : 'destino_ida',
    'place_origin_return_legend'          : 'origem_volta',
    'place_destination_return_legend'     : 'destino_volta',
    'fk_departure_ota_bus_company_legend' : 'empresa_ida',
    'fk_return_ota_bus_company_legend'    : 'empresa_volta',
    'total_tickets_quantity_success'      : 'quantidade_tickets',
})

print(f"Colunas finais ({df_limpo.shape[1]}): {list(df_limpo.columns)}")

Colunas finais (13): ['nk_ota_localizer_id', 'data_compra', 'time_purchase', 'gmv_success', 'quantidade_tickets', 'origem_ida', 'destino_ida', 'origem_volta', 'destino_volta', 'empresa_ida', 'empresa_volta', 'id_cliente', 'periodo_covid']


In [7]:
df_limpo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1741344 entries, 0 to 1741343
Data columns (total 13 columns):
 #   Column               Dtype         
---  ------               -----         
 0   nk_ota_localizer_id  object        
 1   data_compra          datetime64[ns]
 2   time_purchase        object        
 3   gmv_success          float64       
 4   quantidade_tickets   int64         
 5   origem_ida           object        
 6   destino_ida          object        
 7   origem_volta         object        
 8   destino_volta        object        
 9   empresa_ida          object        
 10  empresa_volta        object        
 11  id_cliente           object        
 12  periodo_covid        object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(10)
memory usage: 172.7+ MB


In [8]:
total_linhas = len(df_limpo)
total_localizadores = df_limpo['nk_ota_localizer_id'].nunique()
print(f"Total de linhas: {total_linhas}")
print(f"Total de localizadores únicos: {total_localizadores}")

Total de linhas: 1741344
Total de localizadores únicos: 1741344


## Limpeza de Registros Inconsistentes

Critérios de remoção (regras de negócio):
1. GMV negativo ou zero — erro de sistema
2. Origem igual ao destino — viagem inválida
3. Destino da volta diferente da origem da ida — inconsistência de ida/volta
4. Ticket com volta preenchida — impossível fisicamente

In [9]:
linhas_antes = len(df_limpo)
removidos = {}

mask = df_limpo['gmv_success'] <= 0
removidos['gmv_negativo_ou_zero'] = mask.sum()
df_limpo = df_limpo[~mask]

mask = df_limpo['origem_ida'] == df_limpo['destino_ida']
removidos['origem_igual_destino'] = mask.sum()
df_limpo = df_limpo[~mask]

mask = (df_limpo['origem_ida'] != df_limpo['destino_volta']) & (df_limpo['destino_volta'] != '0')
removidos['inconsistencia_ida_volta'] = mask.sum()
df_limpo = df_limpo[~mask]

mask = (df_limpo['quantidade_tickets'] == 1) & (df_limpo['destino_volta'] != '0')
removidos['ticket_unico_com_volta'] = mask.sum()
df_limpo = df_limpo[~mask]

print("--- Resumo da Limpeza ---")
for motivo, qtd in removidos.items():
    print(f"  {motivo:<30}: {qtd:>6} linhas")
print(f"  {'─'*42}")
print(f"  Total removido                  : {linhas_antes - len(df_limpo):>6}")
print(f"  Total restante                  : {len(df_limpo):>6}")

--- Resumo da Limpeza ---
  gmv_negativo_ou_zero          :    185 linhas
  origem_igual_destino          :  14742 linhas
  inconsistencia_ida_volta      :   2350 linhas
  ticket_unico_com_volta        :     31 linhas
  ──────────────────────────────────────────
  Total removido                  :  17308
  Total restante                  : 1724036


## Ordenação Temporal

Necessário para garantir que o split não cause data leakage.

O comportamento de compra é temporal. Padrões de consumo, popularidade de trechos e até o efeito de campanhas de marketing mudam ao longo do tempo.

In [10]:
df_limpo = df_limpo.sort_values('data_compra').reset_index(drop=True)
print(f"Dados ordenados por data_compra.")
print(f"Primeiro registro: {df_limpo['data_compra'].min().date()}")
print(f"Último registro  : {df_limpo['data_compra'].max().date()}")

Dados ordenados por data_compra.
Primeiro registro: 2013-09-12
Último registro  : 2024-04-01


### Feriado

In [11]:
from src.utils import enriquecer_dados_temporais

df_limpo = enriquecer_dados_temporais(df_limpo, col_data='data_compra', dias_antecedencia=5)

df_limpo['hora_compra'] = pd.to_datetime(df_limpo['time_purchase'], format='%H:%M:%S', errors='coerce').dt.hour

bins = [-1, 5, 11, 17, 24]
labels = ['Madrugada', 'Manhã', 'Tarde', 'Noite']
df_limpo['periodo_compra'] = pd.cut(df_limpo['hora_compra'], bins=bins, labels=labels, right=True)

print(df_limpo['compra_ate_5_dias_feriado'].value_counts(normalize=True).round(3) * 100)

compra_ate_5_dias_feriado
0    82.3
1    17.7
Name: proportion, dtype: float64


### Trecho e Empresa

In [12]:
df_limpo['trecho_ida'] = df_limpo['origem_ida'] + ' -> ' + df_limpo['destino_ida']

df_limpo['trecho_volta_valido'] = df_limpo.apply(
    lambda x: x['origem_volta'] + ' -> ' + x['destino_volta'] if x['destino_volta'] != '0' else 'Sem Volta',
    axis=1
)

df_limpo['mesma_empresa_ida_volta'] = (
    (df_limpo['empresa_ida'] == df_limpo['empresa_volta']) &
    (df_limpo['empresa_volta'] != '0')
).astype(int)

df_limpo['mesmo_trecho_ida_volta'] = (
    (df_limpo['origem_ida'] == df_limpo['destino_volta']) &
    (df_limpo['destino_ida'] == df_limpo['origem_volta']) &
    (df_limpo['destino_volta'] != '0')
).astype(int)

df_limpo['combinacao_empresa'] = df_limpo['empresa_ida'] + ' + ' + df_limpo['empresa_volta']

### Faixa de tickets

In [13]:
df_limpo['faixa_tickets'] = pd.cut(
    df_limpo['quantidade_tickets'],
    bins=[0, 1, 2, 3, 5, 100],
    labels=['1 Ticket', '2 Tickets', '3 Tickets', '4-5 Tickets', '6+ Tickets']
)

## Otimização e Separação dos Dados

In [14]:
df_limpo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1724036 entries, 0 to 1724035
Data columns (total 28 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   nk_ota_localizer_id        object        
 1   data_compra                datetime64[ns]
 2   time_purchase              object        
 3   gmv_success                float64       
 4   quantidade_tickets         int64         
 5   origem_ida                 object        
 6   destino_ida                object        
 7   origem_volta               object        
 8   destino_volta              object        
 9   empresa_ida                object        
 10  empresa_volta              object        
 11  id_cliente                 object        
 12  periodo_covid              object        
 13  ano                        int16         
 14  mes                        int8          
 15  dia                        int8          
 16  dia_semana                 int8     

In [15]:
df_limpo['mesma_empresa_ida_volta'] = df_limpo['mesma_empresa_ida_volta'].astype('int8')
df_limpo['mesmo_trecho_ida_volta'] = df_limpo['mesmo_trecho_ida_volta'].astype('int8')
df_limpo['quantidade_tickets'] = df_limpo['quantidade_tickets'].astype('int16')

In [16]:
from src.utils import split_temporal

df_treino, df_val, df_teste = split_temporal(
    df=df_limpo,
    test_size=340000,
    val_size=170000,
    col_data='data_compra'
)

Treino:    1214036 linhas
Validação: 170000 linhas
Teste:     340000 linhas


In [17]:
DRIVE_BASE = "/content/drive/MyDrive/Portifólio DS Vini/Challenge_ClickBus_2025/data"

salvar_mapeamentos(dicionarios, f"{DRIVE_BASE}/prata/mapeamentos.json")
df_limpo.to_parquet(f"{DRIVE_BASE}/prata/clickbus_silver_completa.parquet", index=False)

df_treino.to_parquet(f"{DRIVE_BASE}/prata/clickbus_treino.parquet", index=False)
df_val.to_parquet(f"{DRIVE_BASE}/prata/clickbus_val.parquet", index=False)
df_teste.to_parquet(f"{DRIVE_BASE}/prata/clickbus_teste.parquet", index=False)

Mapeamentos salvos em: /content/drive/MyDrive/Portifólio DS Vini/Challenge_ClickBus_2025/data/prata/mapeamentos.json
